# 阶段二：统一工具接口

本 Demo 用一个工具注册表和统一的 `ToolCall` / `ToolResult` 表示不同工具。
本阶段暂不处理并发、超时、取消和队列压力。

In [ ]:
from dataclasses import dataclass
from typing import Any, Awaitable, Callable


@dataclass
class ToolCall:
    """描述一次工具调用。"""

    name: str
    arguments: dict[str, Any]
    kind: str = "read"


@dataclass
class ToolResult:
    """统一表示工具调用结果。"""

    status: str
    value: Any = None
    error: str | None = None


Tool = Callable[..., Awaitable[Any]]
"""
Tool 代表“一个可以接收任意参数，并且异步返回任意结果的函数”。
Callable: 表示“可调用对象”，通常就是函数。
...: 表示参数不限制数量和类型
Awaitable[Any]: 表示函数返回一个“可以被 await 的对象”，最终结果类型是 Any。

Callable 的类型参数规定为：Callable[[参数类型1, 参数类型2], 返回值类型]
- [...]：参数类型列表；... 表示任意参数
- ()：函数调用语法，不用于 Callable 的类型参数

如果表示“不接收参数”，应写：
Callable[[], Awaitable[Any]]

"""

async def search(query: str) -> list[str]:
    """模拟搜索工具。

    参数:
        query: 要搜索的关键词。

    返回:
        模拟的搜索结果列表。
    """
    return [f"result for {query}"]


async def fetch_weather(city: str) -> dict[str, str]:
    """模拟天气查询工具。

    参数:
        city: 要查询天气的城市。

    返回:
        包含城市和天气描述的字典。
    """
    return {"city": city, "weather": "sunny"}


# 工具表映射
def register_tool(registry: dict[str, Tool], name: str, tool: Tool) -> None:
    """将工具注册到工具表。

    参数:
        registry: 工具名称到异步工具函数的映射。
        name: 模型或调用方使用的工具名称。
        tool: 接收关键字参数并返回可等待结果的异步函数。
    """
    registry[name] = tool


async def execute_tool(registry: dict[str, Tool], call: ToolCall) -> ToolResult:
    """按统一调用协议执行工具并包装结果。

    参数:
        registry: 已注册工具的名称映射。
        call: 包含工具名、参数和工具类型的调用描述。

    返回:
        成功时返回 `success`；工具不存在或执行报错时返回 `failed`。
    """
    
    # 校验tool的存在性
    tool = registry.get(call.name)
    if tool is None:
        return ToolResult(status="failed", error=f"unknown tool: {call.name}")

    try:
        value = await tool(**call.arguments)
    except (TypeError, ValueError) as exc:
        return ToolResult(status="failed", error=str(exc))
    return ToolResult(status="success", value=value)
"""
as exc 不是必须的；如果不需要读取异常信息

except Exception: (这是默认捕获所有异常)
KeyError      # 字典中没有这个键
IndexError    # 列表下标越界
AttributeError # 对象没有这个属性
NameError     # 变量未定义
FileNotFoundError # 文件不存在
ZeroDivisionError # 除零
RuntimeError  # 运行时错误
OSError       # 操作系统相关错误
"""

async def main() -> None:
    """演示不同工具如何通过同一个入口执行。"""
    registry: dict[str, Tool] = {}
    register_tool(registry, "search", search)
    register_tool(registry, "fetch_weather", fetch_weather)

    # 这次要执行的工具调用请求
    calls = [
        ToolCall("search", {"query": "asyncio"}),
        ToolCall("fetch_weather", {"city": "Shanghai"}),
        ToolCall("missing", {}),
    ]

    results = []
    for call in calls:
        result = await execute_tool(registry, call)
        results.append(result)
        print(call.name, "->", result)
    assert results[0].status == "success"
    assert results[1].status == "success"
    assert results[2].status == "failed"


await main()

search -> ToolResult(status='success', value=['result for asyncio'], error=None)
fetch_weather -> ToolResult(status='success', value={'city': 'Shanghai', 'weather': 'sunny'}, error=None)
missing -> ToolResult(status='failed', value=None, error='unknown tool: missing')


In [ ]:
from langchain_core.runnables import RunnableLambda

async def langchain_async_example() -> None:
    """使用 LangChain Runnable 的 `ainvoke` 调用统一工具入口。"""
    async def run(call: ToolCall) -> ToolResult:
        return await execute_tool(registry, call)

    runnable = RunnableLambda(afunc=run)
    result = await runnable.ainvoke(ToolCall("search", {"query": "asyncio"}))
    print("LangChain Async:", result)
    assert result.status == "success"

await langchain_async_example()


In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.post("/phase2/tools")
async def execute_phase2_tools() -> list[ToolResult]:
    """提供 HTTP 入口，执行阶段二的统一工具调用。"""
    registry: dict[str, Tool] = {}
    register_tool(registry, "search", search)
    return [await execute_tool(registry, ToolCall("search", {"query": "asyncio"}))]

assert any(route.path == "/phase2/tools" for route in app.routes)
print("FastAPI route registered")
